# VIIRS NTL 臺灣範圍裁切工具

用途：把大型 VIIRS GeoTIFF（例如 `avg_rade9h.tif`、`cf_cvg.tif`）裁成臺灣研究範圍，並使用無損 DEFLATE 壓縮輸出。

研究範圍：118–122.5°E、21.5–26.5°N。


## 1. 安裝套件
先執行下一格。

In [1]:
!pip -q install rasterio


## 2. 掛載 Google Drive
執行後，按照畫面授權你的 Google Drive。

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. 設定檔案位置
請先在 Google Drive 建立資料夾：`MyDrive/UVI_NTL/raw/`

把原始檔放進去後，確認檔名與下面一致；若你的檔名不同，只要修改 `input_files` 裡的名稱。

In [5]:
from pathlib import Path

BASE = Path('/content/drive/MyDrive/UVI_NTL')
RAW = BASE / 'raw'
OUT = BASE / 'output'
OUT.mkdir(parents=True, exist_ok=True)

input_files = [
    RAW / 'SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.avg_rade9h.tif',
    RAW / 'SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.cf_cvg.tif',
]

for p in input_files:
    print(p, '存在' if p.exists() else '找不到')

/content/drive/MyDrive/UVI_NTL/raw/SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.avg_rade9h.tif 存在
/content/drive/MyDrive/UVI_NTL/raw/SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.cf_cvg.tif 存在


## 4. 裁切 + 無損壓縮
這格會保留原始像素值、CRS、NoData，並用 DEFLATE 壓縮。

In [7]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# 臺灣研究範圍（WGS84 經緯度）
WGS84_BOUNDS = (118.0, 21.5, 122.5, 26.5)

def clip_geotiff(src_path, dst_path):
    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError(
                f'{src_path.name} 沒有 CRS，先不要繼續，請把畫面貼給 ChatGPT。'
            )

        print(f'正在處理：{src_path.name}')
        print(f'原始 CRS：{src.crs}')
        print(f'原始尺寸：{src.width} x {src.height}')

        # 將臺灣 WGS84 範圍轉成原始 raster 的 CRS
        bounds_in_src = transform_bounds(
            'EPSG:4326',
            src.crs,
            *WGS84_BOUNDS,
            densify_pts=21
        )

        window = from_bounds(
            *bounds_in_src,
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        # 與原始 raster 範圍取交集
        full_window = rasterio.windows.Window(
            0, 0, src.width, src.height
        )

        window = window.intersection(full_window)

        # 讀取臺灣範圍
        data = src.read(window=window)

        out_transform = src.window_transform(window)

        # 複製原始 metadata
        profile = src.profile.copy()

        # 移除原始檔可能不符合新版 GDAL 規定的 block 設定
        profile.pop('blockxsize', None)
        profile.pop('blockysize', None)

        # 更新輸出設定
        profile.update(
            height=data.shape[1],
            width=data.shape[2],
            transform=out_transform,

            # 無損壓縮
            compress='DEFLATE',

            # GeoTIFF 分塊
            tiled=True,
            blockxsize=256,
            blockysize=256,

            BIGTIFF='IF_SAFER'
        )

        # 浮點 raster
        if data.dtype.kind == 'f':
            profile['predictor'] = 3

        # 整數 raster
        elif data.dtype.kind in ('u', 'i'):
            profile['predictor'] = 2

        # 寫出 GeoTIFF
        with rasterio.open(
            dst_path,
            'w',
            **profile
        ) as dst:
            dst.write(data)

        print(f'完成：{dst_path}')
        print(f'輸出尺寸：{data.shape[2]} x {data.shape[1]} pixels')
        print(f'資料型態：{data.dtype}')
        print('-' * 60)


# 依序處理兩個檔案
for src_path in input_files:

    if not src_path.exists():
        print(f'略過，找不到：{src_path}')
        continue

    if 'avg_rade9h' in src_path.name:
        dst_path = OUT / 'Taiwan_NTL_2023_11_avg_rade9h.tif'

    elif 'cf_cvg' in src_path.name:
        dst_path = OUT / 'Taiwan_NTL_2023_11_cf_cvg.tif'

    else:
        dst_path = OUT / f'Taiwan_{src_path.name}'

    clip_geotiff(src_path, dst_path)

正在處理：SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.avg_rade9h.tif
原始 CRS：EPSG:4326
原始尺寸：28800 x 18000
完成：/content/drive/MyDrive/UVI_NTL/output/Taiwan_NTL_2023_11_avg_rade9h.tif
輸出尺寸：1080 x 1200 pixels
資料型態：float32
------------------------------------------------------------
正在處理：SVDNB_npp_20231101-20231130_75N060E_vcmcfg_v10_c202312080900.cf_cvg.tif
原始 CRS：EPSG:4326
原始尺寸：28800 x 18000
完成：/content/drive/MyDrive/UVI_NTL/output/Taiwan_NTL_2023_11_cf_cvg.tif
輸出尺寸：1080 x 1200 pixels
資料型態：uint16
------------------------------------------------------------


## 5. 檢查輸出檔大小
如果輸出已低於 512 MB，就可以直接上傳到 ChatGPT。

In [8]:
for p in OUT.glob('*.tif'):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'{p.name}: {size_mb:.2f} MB')


Taiwan_NTL_2023_11_avg_rade9h.tif: 3.12 MB
Taiwan_NTL_2023_11_cf_cvg.tif: 0.38 MB


## 6. 基本驗證
這格會顯示輸出檔的 CRS、範圍、解析度、NoData，方便確認沒有跑掉。

In [9]:
for p in OUT.glob('*.tif'):
    with rasterio.open(p) as ds:
        print('\n', p.name)
        print('CRS:', ds.crs)
        print('Bounds:', ds.bounds)
        print('Resolution:', ds.res)
        print('NoData:', ds.nodata)
        print('Count:', ds.count)
        print('Dtype:', ds.dtypes)



 Taiwan_NTL_2023_11_avg_rade9h.tif
CRS: EPSG:4326
Bounds: BoundingBox(left=117.99791713065, bottom=21.502082905349997, right=122.49791716665, top=26.502082945349997)
Resolution: (0.0041666667, 0.0041666667)
NoData: None
Count: 1
Dtype: ('float32',)

 Taiwan_NTL_2023_11_cf_cvg.tif
CRS: EPSG:4326
Bounds: BoundingBox(left=117.99791713065, bottom=21.502082905349997, right=122.49791716665, top=26.502082945349997)
Resolution: (0.0041666667, 0.0041666667)
NoData: None
Count: 1
Dtype: ('uint16',)
